# Which Clusters Change the Most from Init to Trained MFA?

Companion to `kmeans_init_vs_mfa_comparison.ipynb`, focused on one question:
**per cluster, how many samples change membership between the k-means init
partition and the trained MFA partition?**

Inputs (same token stream, same K, identity cluster correspondence):

- **k-means (init)**: `kmeans_centroid_assignments.pt` — nearest Euclidean
  centroid per token.
- **MFA (trained)**: `mfa_model_assignments.pt` — argmax-responsibility
  component per token.

For each cluster `k` we count, from the K×K contingency matrix:

- `left_k`   = samples in k-means cluster k that end up elsewhere under MFA
- `joined_k` = samples in MFA cluster k that came from another init cluster
- `churn_k`  = `left_k + joined_k` — the **membership-change count**
  (size of the symmetric difference `km_k Δ mfa_k`)
- `net_delta_k` = `mfa_size_k − km_size_k` — the net size change

and plot the distributions of these deltas, raw and normalized by cluster size.


## 1. Setup and Artifact Validation

In [1]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception as exc:
    px = None
    go = None
    print(f"Plotly unavailable: {exc}")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/dalg").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO = find_repo_root()

# ---------------------------------------------------------------- parameters
LAYER = 5
K = 1000
Q = 394
EPOCHS = 15
# ------------------------------------------------- paths built from parameters
MFA_RUN = REPO / f"dalg-cache/pile_gemma2b_models/layer{LAYER:02d}_{K}_{Q}_component_sharded_mfa"
MFA_ASSIGN_PATH = MFA_RUN / "mfa_model_assignments.pt"
MFA_INIT_CENTROIDS = MFA_RUN / "centroids.pt"

CENTROIDS_DIR = REPO / f"dalg-cache/pile_gemma2b_models/centroids/k{K}_L{LAYER:02d}"
KMEANS_CENTROIDS = CENTROIDS_DIR / "centroids.pt"
KMEANS_ASSIGN = CENTROIDS_DIR / "kmeans_centroid_assignments.pt"

# ------------------------------------------------------------- plot constants
COLOR_KMEANS = "#2a78d6"  # blue  — k-means init partition
COLOR_MFA = "#1baf7a"     # aqua  — trained MFA partition
COLOR_NEUTRAL = "#52514e" # gray  — derived quantities (deltas)
PLOT_TEMPLATE = "plotly_white"

PLOTS_DIR = REPO / "notebooks/plots"
PLOTS_HTML_DIR = PLOTS_DIR / "html"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_HTML_DIR.mkdir(parents=True, exist_ok=True)
PLOT_TAG = f"L{LAYER:02d}_K{K}_q{Q}_ep{EPOCHS}_membership"


def save_fig(fig, name: str) -> None:
    fig.update_layout(title_font_size=13, title_x=0.5, margin=dict(t=48))
    path = PLOTS_DIR / f"{name}_{PLOT_TAG}.pdf"
    try:
        fig.write_image(str(path), width=1000, height=550)
    except Exception as exc:
        path = PLOTS_HTML_DIR / f"{name}_{PLOT_TAG}.html"
        fig.write_html(str(path), include_plotlyjs="cdn")
        print(f"PDF export failed ({exc}); saved {path.name} instead")


artifacts = pd.DataFrame(
    [
        {"artifact": "kmeans assignments", "path": str(KMEANS_ASSIGN), "exists": KMEANS_ASSIGN.exists()},
        {"artifact": "mfa assignments", "path": str(MFA_ASSIGN_PATH), "exists": MFA_ASSIGN_PATH.exists()},
        {"artifact": "kmeans centroids", "path": str(KMEANS_CENTROIDS), "exists": KMEANS_CENTROIDS.exists()},
        {"artifact": "mfa init centroids", "path": str(MFA_INIT_CENTROIDS), "exists": MFA_INIT_CENTROIDS.exists()},
    ]
)
display(artifacts)
assert KMEANS_ASSIGN.exists() and MFA_ASSIGN_PATH.exists(), "missing assignment artifacts"


,artifact,path,exists
0,kmeans assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
1,mfa assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
2,kmeans centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
3,mfa init centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True


### 1.1 Cluster Correspondence Check

Per-cluster deltas only make sense if cluster id `k` means the same thing in
both partitions, i.e. the nearest-centroid assignments used the exact centroids
the MFA was initialized from. Check bit-identity; if it fails, do not trust the
identity mapping below.


In [2]:
def _centroid_tensor(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict):
        for key in ("centroids", "mu", "means"):
            if key in obj:
                obj = obj[key]
                break
    return obj.float()


if KMEANS_CENTROIDS.exists() and MFA_INIT_CENTROIDS.exists():
    c_km = _centroid_tensor(KMEANS_CENTROIDS)
    c_init = _centroid_tensor(MFA_INIT_CENTROIDS)
    identity_ok = c_km.shape == c_init.shape and torch.equal(c_km, c_init)
    print(f"kmeans centroids: {tuple(c_km.shape)}, mfa init centroids: {tuple(c_init.shape)}")
    print(f"bit-identical: {identity_ok}")
    if not identity_ok:
        print("WARNING: centroids differ -> per-cluster deltas below are NOT meaningful under identity mapping.")
    del c_km, c_init
else:
    print("Skipped: missing centroid file(s); identity correspondence UNVERIFIED.")


kmeans centroids: (1000, 2048), mfa init centroids: (1000, 2048)
bit-identical: True


## 2. Load Assignments and Build the Contingency Matrix

Everything per-cluster derives from the K×K contingency matrix
`C[i, j] = #tokens with kmeans id i and mfa id j`, computed in one `bincount`
pass so both ~N-length assignment vectors can be freed immediately.


In [3]:
def load_assignments(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    assert int(obj["K"]) == K, (obj["K"], K)
    return obj["assignments"].to(torch.long)


a_km = load_assignments(KMEANS_ASSIGN)
a_mfa = load_assignments(MFA_ASSIGN_PATH)
assert a_km.numel() == a_mfa.numel(), (a_km.numel(), a_mfa.numel())
N = a_km.numel()
print(f"Loaded {N:,} token assignments for both partitions")

contingency = (
    torch.bincount(a_km * K + a_mfa, minlength=K * K).reshape(K, K).numpy().astype(np.int64)
)
del a_km, a_mfa
gc.collect()

km_sizes = contingency.sum(axis=1)
mfa_sizes = contingency.sum(axis=0)
shared = np.diag(contingency).copy()

same_id_agreement = shared.sum() / N
print(f"same-id agreement: {same_id_agreement:.4f} "
      f"({N - shared.sum():,} of {N:,} tokens changed cluster)")


Loaded 73,687,936 token assignments for both partitions
same-id agreement: 0.4071 (43,689,840 of 73,687,936 tokens changed cluster)


### 2.1 Cluster Size Distribution

Compare the number of tokens assigned to each cluster before and after MFA
training. Both distributions use the same log-spaced bins.


In [4]:
if go is not None:
    positive_sizes = np.concatenate([km_sizes[km_sizes > 0], mfa_sizes[mfa_sizes > 0]])
    edges = np.logspace(np.log10(positive_sizes.min()), np.log10(positive_sizes.max()), 61)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)

    fig = go.Figure()
    for sizes, name, color in [
        (km_sizes, "k-means (init)", COLOR_KMEANS),
        (mfa_sizes, "MFA (trained)", COLOR_MFA),
    ]:
        counts, _ = np.histogram(sizes[sizes > 0], bins=edges)
        fig.add_bar(x=centers, y=counts, width=widths, name=name, marker_color=color, opacity=0.6)

    fig.update_xaxes(type="log")
    fig.update_layout(
        title="Cluster size distribution",
        xaxis_title="assigned tokens per cluster (log-spaced bins)",
        yaxis_title="clusters",
        template=PLOT_TEMPLATE,
        barmode="overlay",
        bargap=0.05,
        width=1000,
        height=550,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    )
    save_fig(fig, "cluster_size_hist")
    fig.show()

    print(f"Zero-size clusters: k-means={(km_sizes == 0).sum()}, MFA={(mfa_sizes == 0).sum()}")


Zero-size clusters: k-means=0, MFA=378


## 3. Per-Cluster Membership Change

For each cluster id `k` (identity mapping):

- `left = km_size − shared`: tokens that were in init cluster k but are
  assigned elsewhere by the trained MFA;
- `joined = mfa_size − shared`: tokens the trained cluster k gained from other
  init clusters;
- `churn = left + joined`: total membership changes touching cluster k;
- `net_delta = mfa_size − km_size = joined − left`: net growth/shrinkage;
- normalized versions divide by the init cluster size (`rel_*`), so a
  `rel_churn` of 1 means as many tokens moved as the cluster originally held.


In [5]:
per_cluster = pd.DataFrame(
    {
        "cluster": np.arange(K),
        "km_size": km_sizes,
        "mfa_size": mfa_sizes,
        "shared": shared,
    }
)
per_cluster["left"] = per_cluster["km_size"] - per_cluster["shared"]
per_cluster["joined"] = per_cluster["mfa_size"] - per_cluster["shared"]
per_cluster["churn"] = per_cluster["left"] + per_cluster["joined"]
per_cluster["net_delta"] = per_cluster["mfa_size"] - per_cluster["km_size"]
denom = np.maximum(per_cluster["km_size"], 1)
per_cluster["rel_left"] = per_cluster["left"] / denom
per_cluster["rel_churn"] = per_cluster["churn"] / denom
per_cluster["rel_net_delta"] = per_cluster["net_delta"] / denom

display(
    per_cluster[["km_size", "mfa_size", "left", "joined", "churn", "net_delta", "rel_churn"]]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)


,km_size,mfa_size,left,joined,churn,net_delta,rel_churn
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,73687.936000,73687.936000,43689.840000,43689.840000,87379.680000,0.000000,1.457057
std,57194.461082,94547.071198,45922.451462,77655.223352,93112.627535,87226.424424,1.986911
min,413.000000,0.000000,0.000000,0.000000,5.000000,-333782.000000,0.005375
10%,22326.100000,0.000000,1840.900000,0.000000,14655.000000,-86657.300000,0.300086
25%,35702.750000,0.000000,8274.750000,0.000000,30939.750000,-46085.750000,0.772614
50%,56763.500000,48965.000000,31029.000000,9905.000000,60527.000000,-4068.500000,1.000000
75%,92937.500000,112982.000000,64920.500000,58297.000000,109474.000000,30828.000000,1.337136
90%,147550.400000,180763.900000,101297.900000,119236.000000,184546.900000,84523.600000,2.741839
max,561332.000000,731406.000000,333782.000000,729597.000000,908115.000000,646663.000000,27.126710


### 3.1 Clusters That Change the Most

Top movers by raw churn (dominated by big clusters) and by relative churn
(membership turnover regardless of size).


In [6]:
cols = ["cluster", "km_size", "mfa_size", "shared", "left", "joined", "churn", "net_delta", "rel_churn"]
display(Markdown("**Top 15 by raw churn (samples changing membership):**"))
display(per_cluster.sort_values("churn", ascending=False)[cols].head(15))

display(Markdown("**Top 15 by relative churn (churn / init size):**"))
display(per_cluster.sort_values("rel_churn", ascending=False)[cols].head(15))


**Top 15 by raw churn (samples changing membership):**

,cluster,km_size,mfa_size,shared,left,joined,churn,net_delta,rel_churn
833,833,180327,731406,1809,178518,729597,908115,551079,5.035935
475,475,46266,692929,80,46186,692849,739035,646663,15.973609
870,870,213391,498440,2,213389,498438,711827,285049,3.335787
604,604,165305,487120,0,165305,487120,652425,321815,3.946795
839,839,67545,557517,9032,58513,548485,606998,489972,8.986572
41,41,337056,231215,12571,324485,218644,543129,-105841,1.611391
95,95,71296,469196,0,71296,469196,540492,397900,7.580958
500,500,166489,372508,0,166489,372508,538997,206019,3.237433
383,383,561332,474751,252078,309254,222673,531927,-86581,0.947616
821,821,48862,468119,1192,47670,466927,514597,419257,10.531640


**Top 15 by relative churn (churn / init size):**

,cluster,km_size,mfa_size,shared,left,joined,churn,net_delta,rel_churn
847,847,15571,407327,254,15317,407073,422390,391756,27.126710
779,779,17045,306066,80,16965,305986,322951,289021,18.946964
7,7,18785,332408,3,18782,332405,351187,313623,18.695076
475,475,46266,692929,80,46186,692849,739035,646663,15.973609
187,187,25249,372051,19280,5969,352771,358740,346802,14.208087
454,454,5577,68110,726,4851,67384,72235,62533,12.952304
257,257,31439,389094,29320,2119,359774,361893,357655,11.510958
0,0,36560,382997,0,36560,382997,419557,346437,11.475848
289,289,21057,225012,2779,18278,222233,240511,203955,11.421902
307,307,26046,265238,2158,23888,263080,286968,239192,11.017738


## 4. Distribution of the Membership-Change Delta

Histograms over the K clusters. `churn` is heavy-tailed, so it is shown on a
log-x axis alongside the size-normalized version; `net_delta` shows whether
training grew or shrank each cluster.


In [7]:
if px is not None:
    # px.histogram(log_x=True) bins linearly and only log-scales the axis, which
    # squashes everything into a few invisible bars; bin in log space instead.
    # churn, left and joined share bins so the three profiles are comparable.
    both = np.concatenate([per_cluster[c].to_numpy() for c in ("churn", "left", "joined")])
    both = np.clip(both, 1, None)
    edges = np.logspace(np.log10(both.min()), np.log10(both.max()), 61)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)

    fig = go.Figure()
    for col, name, color in [
        ("churn", "churn (left + joined)", COLOR_NEUTRAL),
        ("left", "left (tokens lost)", COLOR_KMEANS),
        ("joined", "joined (tokens gained)", COLOR_MFA),
    ]:
        counts, _ = np.histogram(np.clip(per_cluster[col].to_numpy(), 1, None), bins=edges)
        fig.add_bar(x=centers, y=counts, width=widths, name=name, marker_color=color, opacity=0.6)
    fig.update_xaxes(type="log")
    fig.update_layout(
        title="Per-cluster membership change (churn = samples that left + samples that joined)",
        xaxis_title="tokens (log-spaced bins)",
        yaxis_title="clusters",
        template=PLOT_TEMPLATE,
        barmode="overlay",
        bargap=0.05,
        width=1000,
        height=600,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    )
    save_fig(fig, "churn_hist")
    fig.show()

    fig = px.histogram(
        per_cluster,
        x="rel_churn",
        nbins=60,
        title="Per-cluster relative membership change (churn / init cluster size)",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
    )
    fig.add_vline(x=1.0, line_dash="dash", line_color="#0b0b0b",
                  annotation_text="churn = init size", annotation_position="top right")
    fig.update_layout(xaxis_title="relative churn", yaxis_title="clusters", bargap=0.05)
    save_fig(fig, "rel_churn_hist")
    fig.show()



### 4.1 Init → Trained Flow Heatmap

Cell `(i, j)` counts the tokens that **left** k-means cluster `i` and
**joined** MFA cluster `j` — i.e. the off-diagonal contingency entry
`C[i, j]`. The diagonal (tokens that stayed in their cluster) is zeroed so it
does not swamp the color scale, and counts are shown as `log10(tokens + 1)`.
Hot rows are init clusters that training drained broadly; hot columns are
trained clusters that absorbed tokens from many init clusters.


In [8]:
# if go is not None:
#     flow = contingency.astype(np.float32).copy()
#     np.fill_diagonal(flow, 0.0)  # diagonal = retained tokens, not membership change
#     fig = go.Figure(
#         go.Heatmap(
#             z=np.log10(flow + 1.0),
#             colorscale="Viridis",
#             zmin=0,
#             zmax=4,
#             colorbar=dict(title="log10(tokens + 1)"),
#             hovertemplate="kmeans i=%{y}<br>mfa j=%{x}<br>log10(tokens+1)=%{z:.2f}<extra></extra>",
#         )
#     )
#     fig.update_layout(
#         title="Membership flow: tokens leaving k-means cluster i and joining MFA cluster j",
#         xaxis_title="MFA cluster j (trained)",
#         yaxis_title="k-means cluster i (init)",
#         yaxis=dict(autorange="reversed"),
#         template=PLOT_TEMPLATE,
#         width=820,
#         height=780,
#     )
#     save_fig(fig, "flow_heatmap")
#     fig.show()
#     del flow


### 4.3 Clustered Flow Heatmap (clustermap)

Same matrix as 4.2, but rows and columns are reordered by hierarchical
clustering (average linkage, Euclidean distance on the `log10(tokens+1)`
flow profiles) so that init clusters draining to similar destinations sit
next to each other, and trained clusters absorbing from similar sources sit
next to each other. Axis positions no longer correspond to cluster ids —
hover to see the true `(i, j)` pair. Blocks of bright cells are groups of
init clusters whose tokens training redistributed into a common set of
trained clusters.


In [9]:
# if go is not None:
#     from scipy.cluster.hierarchy import leaves_list, linkage

#     flow = contingency.astype(np.float32).copy()
#     np.fill_diagonal(flow, 0.0)
#     log_flow = np.log10(flow + 1.0)

#     # euclidean rather than cosine: clusters with zero outflow/inflow produce
#     # all-zero profiles, for which cosine distance is undefined
#     row_order = leaves_list(linkage(log_flow, method="average", metric="euclidean"))
#     col_order = leaves_list(linkage(log_flow.T, method="average", metric="euclidean"))
#     z = log_flow[np.ix_(row_order, col_order)]

#     fig = go.Figure(
#         go.Heatmap(
#             z=z,
#             x=[str(j) for j in col_order],
#             y=[str(i) for i in row_order],
#             colorscale="Viridis",
#             zmin=0,
#             zmax=4,
#             colorbar=dict(title="log10(tokens + 1)"),
#             hovertemplate="kmeans i=%{y}<br>mfa j=%{x}<br>log10(tokens+1)=%{z:.2f}<extra></extra>",
#         )
#     )
#     fig.update_layout(
#         title="Clustered membership flow (rows/cols reordered by hierarchical clustering)",
#         xaxis=dict(title="MFA cluster j (trained, clustered order)", showticklabels=False),
#         yaxis=dict(title="k-means cluster i (init, clustered order)", showticklabels=False, autorange="reversed"),
#         template=PLOT_TEMPLATE,
#         width=820,
#         height=780,
#     )
#     save_fig(fig, "flow_clustermap")
#     fig.show()
#     del flow, log_flow, z


## 4.4 Tokens Exchanged vs Centroid Distance

Does a cluster exchange more tokens with clusters that are near it, or far from
it? Each point below is a cluster; we correlate **how many tokens it exchanges**
against the **token-weighted mean centroid distance to its exchange partners**.
Each partner is weighted by the number of tokens actually exchanged with it
(`Σ_p C[k,p]·d(k,p) / Σ_p C[k,p]`), so the y-axis reflects where the token
*mass* goes — not a rare 1-token leak to a distant cluster — and is on the same
token-weighted footing as the x-axis. Stratified into:

- **give**: tokens `k` gave away (`left`) vs distance to partners `j`, weighted by `C[k,j]`;
- **take**: tokens `k` took in (`joined`) vs distance to partners `i`, weighted by `C[i,k]`;
- **give+take**: total `churn` vs distance to all partners, weighted by `C[k,p]+C[p,k]`.

The first cell computes the weighted mean partner distances (under identity
cluster correspondence, with both the **k-means centroids** and the **MFA
means** as cluster centers); the scatter cell plots tokens exchanged (x) against
that distance (y) with a Spearman ρ per stratum.


In [10]:
from scipy.spatial.distance import cdist

from dalg.models.mfa import load_mfa

# cluster centers under identity correspondence
c_km = _centroid_tensor(KMEANS_CENTROIDS).numpy()
_model = load_mfa(MFA_RUN / "mfa_model.pt", map_location="cpu")
mu = _model.mu.detach().float().cpu().numpy()
del _model
gc.collect()
assert c_km.shape == mu.shape == (K, c_km.shape[1])

# off-diagonal flow -> per-partner token weights (match the x-axis token counts)
_flow = contingency.astype(np.float64).copy()
np.fill_diagonal(_flow, 0.0)
W_give = _flow                 # W_give[k, j] = tokens k -> j        (rowsum = left)
W_take = _flow.T               # W_take[k, i] = tokens i -> k        (rowsum = joined)
W_both = W_give + W_take       # total tokens exchanged with partner (rowsum = churn)


def weighted_partner_mean(values: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Token-weighted mean of values[k, p] over partners p (weight = tokens exchanged)."""
    denom = weights.sum(axis=1)
    num = (values * weights).sum(axis=1)
    return np.where(denom > 0, num / np.maximum(denom, 1), np.nan)


exchange = pd.DataFrame({"cluster": np.arange(K)})
exchange["n_give"] = (W_give > 0).sum(axis=1)
exchange["n_take"] = (W_take > 0).sum(axis=1)
exchange["n_both"] = (W_both > 0).sum(axis=1)
for label, centers in [("km", c_km), ("mfa", mu)]:
    D = cdist(centers, centers)
    exchange[f"give_dist_{label}"] = weighted_partner_mean(D, W_give)
    exchange[f"take_dist_{label}"] = weighted_partner_mean(D, W_take)
    exchange[f"both_dist_{label}"] = weighted_partner_mean(D, W_both)

per_cluster = per_cluster.merge(exchange, on="cluster")

display(
    exchange[
        ["n_give", "n_take", "n_both",
         "give_dist_km", "take_dist_km", "both_dist_km",
         "give_dist_mfa", "take_dist_mfa", "both_dist_mfa"]
    ].describe(percentiles=[0.1, 0.5, 0.9])
)


,n_give,n_take,n_both,give_dist_km,take_dist_km,both_dist_km,give_dist_mfa,take_dist_mfa,both_dist_mfa
count,1000.000000,1000.000000,1000.000000,998.000000,619.000000,1000.000000,998.000000,619.000000,1000.000000
mean,101.342000,101.342000,195.066000,38.203083,36.413637,37.509424,33.076279,35.653039,32.642761
std,91.882552,135.485702,131.495318,12.515650,14.867597,12.949986,11.715324,14.613781,12.210580
min,0.000000,0.000000,1.000000,12.329892,10.799641,12.036058,9.740144,12.082666,11.234614
10%,16.900000,0.000000,36.000000,25.963754,22.466574,26.288945,22.410329,22.582920,22.660143
50%,64.000000,23.500000,176.000000,36.500892,34.278084,35.539925,30.421589,32.609646,29.717538
90%,251.100000,321.000000,371.200000,51.135189,49.580263,49.135113,46.204681,49.845877,45.410871
max,422.000000,741.000000,754.000000,120.191320,159.405726,145.043318,120.026639,132.728711,120.026639


In [11]:
if px is not None:
    from scipy.stats import spearmanr

    # per-cluster: x = tokens exchanged, y = weighted mean centroid distance to partners
    strata = [
        ("give", "left", COLOR_KMEANS),        # tokens k gave away
        ("take", "joined", COLOR_MFA),         # tokens k took in
        ("give+take", "churn", COLOR_NEUTRAL), # both
    ]
    for label, title in [("km", "k-means centroid geometry"), ("mfa", "MFA mean geometry")]:
        fig = go.Figure()
        for name, xcol, color in strata:
            ycol = f"{'both' if name == 'give+take' else name}_dist_{label}"
            sub = per_cluster[["cluster", xcol, ycol]].dropna()
            sub = sub[sub[xcol] > 0]
            x, y, cl = sub[xcol].to_numpy(), sub[ycol].to_numpy(), sub["cluster"].to_numpy()
            rho, _ = spearmanr(x, y)
            fig.add_scatter(
                x=x, y=y, mode="markers",
                name=f"{name} (ρ={rho:.2f})",
                marker=dict(color=color, size=5, opacity=0.5),
                customdata=cl,
                hovertemplate="cluster %{customdata}<br>exchanged=%{x:,}<br>distance=%{y:.2f}<extra></extra>",
            )
            # least-squares fit y ~ a*log10(x) + b (linear in the plotted log-x axis)
            lx = np.log10(x)
            a, b = np.polyfit(lx, y, 1)
            xs = np.linspace(lx.min(), lx.max(), 100)
            fig.add_scatter(
                x=10 ** xs, y=a * xs + b, mode="lines",
                line=dict(color=color, width=2),
                name=f"{name} fit", showlegend=False, hoverinfo="skip",
            )
        fig.update_xaxes(type="log")
        fig.update_layout(
            title=f"Tokens exchanged vs mean centroid distance to partners ({title})",
            xaxis_title="tokens exchanged (log)",
            yaxis_title="mean centroid distance to exchange partners",
            template=PLOT_TEMPLATE,
            width=1000,
            height=550,
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )
        save_fig(fig, f"exchange_distance_vs_count_{label}")
        fig.show()


## 4.5 Tokens Exchanged vs Intrinsic-Dimension Gap

Same construction as 4.4, but the y-axis replaces centroid distance with the
token-weighted mean **intrinsic-dimension gap** to the exchange partners: for
cluster `k`, `Σ_p C[k,p]·(ID[p] − ID[k]) / Σ_p C[k,p]` over its partners `p`
(each weighted by tokens exchanged). A positive value means `k`'s exchanged
tokens go mostly to/from clusters of *higher* intrinsic dimension than itself.

Stratified the same way — **give** vs `left`, **take** vs `joined`,
**give+take** vs `churn` (same token weights as 4.4) — and computed for both
intrinsic-dimension sources: the **k-means partition** IDs
(`centroids_1000_05/intrinsic_dims.pt`) and the **MFA partition** IDs
(`1000_05_10/intrinsic_dims.pt`), under identity cluster correspondence.


In [12]:
# intrinsic-dim per cluster from each partition (identity cluster correspondence)
ID_PATHS = {
    "km": CENTROIDS_DIR / "intrinsic_dims.pt",
    "mfa": MFA_RUN / "intrinsic_dims.pt",
}
ids_by_source = {}
for label, path in ID_PATHS.items():
    obj = torch.load(path, map_location="cpu")
    assert int(obj["K"]) == K, (label, obj["K"], K)
    ids_by_source[label] = obj["intrinsic_dims"].to(torch.float64).numpy()


def weighted_partner_id_gap(ids: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Token-weighted mean over partners of (partner_ID - k_ID); ID==0 = invalid partner."""
    valid = ids > 0
    diff = ids[None, :] - ids[:, None]     # diff[k, p] = ID[p] - ID[k]
    w = weights * valid[None, :]           # drop invalid partners from the weighting
    denom = w.sum(axis=1)
    num = (diff * w).sum(axis=1)
    return np.where((denom > 0) & valid, num / np.maximum(denom, 1), np.nan)


# W_give / W_take / W_both come from the 4.4 exchange cell (token weights)
for label, ids in ids_by_source.items():
    per_cluster[f"give_iddiff_{label}"] = weighted_partner_id_gap(ids, W_give)
    per_cluster[f"take_iddiff_{label}"] = weighted_partner_id_gap(ids, W_take)
    per_cluster[f"both_iddiff_{label}"] = weighted_partner_id_gap(ids, W_both)

display(
    per_cluster[
        ["give_iddiff_km", "take_iddiff_km", "both_iddiff_km",
         "give_iddiff_mfa", "take_iddiff_mfa", "both_iddiff_mfa"]
    ].describe(percentiles=[0.1, 0.5, 0.9])
)


,give_iddiff_km,take_iddiff_km,both_iddiff_km,give_iddiff_mfa,take_iddiff_mfa,both_iddiff_mfa
count,998.000000,619.000000,1000.000000,604.000000,599.000000,606.000000
mean,-22.489574,69.684517,-0.460927,23.424080,15.065546,21.317226
std,124.030606,122.613333,135.578266,99.065570,97.710246,95.930174
min,-483.386037,-534.126866,-483.386037,-404.093711,-352.837557,-355.404654
10%,-177.930244,-73.020062,-166.316697,-98.369732,-100.931673,-91.627684
50%,-13.180610,61.890410,7.423377,24.840681,14.384231,22.370132
90%,118.187581,229.713457,155.248735,146.839480,127.201396,142.975483
max,454.727377,569.326027,563.636119,321.343305,359.627594,301.705621


In [13]:
if px is not None:
    from scipy.stats import spearmanr

    strata = [
        ("give", "left", COLOR_KMEANS),
        ("take", "joined", COLOR_MFA),
        ("give+take", "churn", COLOR_NEUTRAL),
    ]
    for label, title in [
        ("km", "k-means partition intrinsic dims"),
        ("mfa", "MFA partition intrinsic dims"),
    ]:
        fig = go.Figure()
        for name, xcol, color in strata:
            ycol = f"{'both' if name == 'give+take' else name}_iddiff_{label}"
            sub = per_cluster[["cluster", xcol, ycol]].dropna()
            sub = sub[sub[xcol] > 0]
            x, y, cl = sub[xcol].to_numpy(), sub[ycol].to_numpy(), sub["cluster"].to_numpy()
            rho, _ = spearmanr(x, y)
            fig.add_scatter(
                x=x, y=y, mode="markers",
                name=f"{name} (ρ={rho:.2f})",
                marker=dict(color=color, size=5, opacity=0.5),
                customdata=cl,
                hovertemplate="cluster %{customdata}<br>exchanged=%{x:,}<br>ID gap=%{y:.1f}<extra></extra>",
            )
            # least-squares fit y ~ a*log10(x) + b (linear in the plotted log-x axis)
            lx = np.log10(x)
            a, b = np.polyfit(lx, y, 1)
            xs = np.linspace(lx.min(), lx.max(), 100)
            fig.add_scatter(
                x=10 ** xs, y=a * xs + b, mode="lines",
                line=dict(color=color, width=2),
                name=f"{name} fit", showlegend=False, hoverinfo="skip",
            )
        fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_xaxes(type="log")
        fig.update_layout(
            title=f"Tokens exchanged vs mean intrinsic-dim gap to partners ({title})",
            xaxis_title="tokens exchanged (log)",
            yaxis_title="mean (partner ID − cluster ID) over exchange partners",
            template=PLOT_TEMPLATE,
            width=1000,
            height=550,
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )
        save_fig(fig, f"exchange_iddiff_vs_count_{label}")
        fig.show()


## 4.6 Outlier Samples in the K-Means Exchange Plots

Each plotted sample is a `(cluster, stratum)` pair, where the stratum is
**give**, **take**, or **give+take**. Within each stratum, outlierness is the
absolute residual from the same linear fit used in the plots,
`y ~ log10(tokens exchanged)`, robustly standardized by the residual median
and median absolute deviation (MAD). The tables show the top
`OUTLIERS_PER_STRATUM` samples from each stratum for the two K-means plots,
followed by samples that are outliers in both plots. A signed residual says
whether the observed value lies above or below its fitted value.


In [14]:
OUTLIERS_PER_STRATUM = 10
OUTLIER_STRATA = [
    ("give", "left", "give_dist_km", "give_iddiff_km"),
    ("take", "joined", "take_dist_km", "take_iddiff_km"),
    ("give+take", "churn", "both_dist_km", "both_iddiff_km"),
]


def ranked_plot_outliers(
    frame: pd.DataFrame,
    *,
    stratum: str,
    x_col: str,
    y_col: str,
    metric_name: str,
) -> pd.DataFrame:
    """Rank samples by robustly standardized residual from y ~ log10(x)."""
    sub = frame[["cluster", x_col, y_col]].dropna().copy()
    sub = sub[sub[x_col] > 0]

    log_x = np.log10(sub[x_col].to_numpy(dtype=np.float64))
    y = sub[y_col].to_numpy(dtype=np.float64)
    slope, intercept = np.polyfit(log_x, y, 1)
    expected = slope * log_x + intercept
    residual = y - expected

    residual_center = np.median(residual)
    mad = np.median(np.abs(residual - residual_center))
    robust_scale = 1.4826 * mad
    if not np.isfinite(robust_scale) or robust_scale <= np.finfo(float).eps:
        robust_scale = np.std(residual, ddof=1)
    if not np.isfinite(robust_scale) or robust_scale <= np.finfo(float).eps:
        robust_scale = 1.0

    sub["stratum"] = stratum
    sub["tokens_exchanged"] = sub[x_col].astype(np.int64)
    sub[f"{metric_name}_expected"] = expected
    sub[f"{metric_name}_residual"] = residual
    sub[f"{metric_name}_outlier_score"] = np.abs(
        (residual - residual_center) / robust_scale
    )
    sub = sub.nlargest(OUTLIERS_PER_STRATUM, f"{metric_name}_outlier_score").copy()
    sub[f"{metric_name}_rank"] = np.arange(1, len(sub) + 1)
    return sub.rename(columns={y_col: metric_name})[
        ["cluster", "stratum", "tokens_exchanged", metric_name,
         f"{metric_name}_expected", f"{metric_name}_residual",
         f"{metric_name}_outlier_score", f"{metric_name}_rank"]
    ]


distance_outliers = pd.concat(
    [
        ranked_plot_outliers(
            per_cluster, stratum=stratum, x_col=x_col, y_col=distance_col,
            metric_name="mean_centroid_distance_km",
        )
        for stratum, x_col, distance_col, _ in OUTLIER_STRATA
    ],
    ignore_index=True,
).sort_values("mean_centroid_distance_km_outlier_score", ascending=False)

id_gap_outliers = pd.concat(
    [
        ranked_plot_outliers(
            per_cluster, stratum=stratum, x_col=x_col, y_col=id_gap_col,
            metric_name="mean_intrinsic_dim_gap_km",
        )
        for stratum, x_col, _, id_gap_col in OUTLIER_STRATA
    ],
    ignore_index=True,
).sort_values("mean_intrinsic_dim_gap_km_outlier_score", ascending=False)

overlap_outliers = distance_outliers.merge(
    id_gap_outliers,
    on=["cluster", "stratum", "tokens_exchanged"],
    how="inner",
)
overlap_outliers["joint_outlier_score"] = (
    overlap_outliers["mean_centroid_distance_km_outlier_score"]
    + overlap_outliers["mean_intrinsic_dim_gap_km_outlier_score"]
)
overlap_outliers = overlap_outliers.sort_values("joint_outlier_score", ascending=False)

display(Markdown("**Outliers: tokens exchanged vs mean centroid distance (K-means geometry)**"))
display(distance_outliers)
display(Markdown("**Outliers: tokens exchanged vs mean intrinsic-dimension gap (K-means partition)**"))
display(id_gap_outliers)
display(Markdown("**Outliers appearing in both K-means plots**"))
display(overlap_outliers)


**Outliers: tokens exchanged vs mean centroid distance (K-means geometry)**

,cluster,stratum,tokens_exchanged,mean_centroid_distance_km,mean_centroid_distance_km_expected,mean_centroid_distance_km_residual,mean_centroid_distance_km_outlier_score,mean_centroid_distance_km_rank
20,485,give+take,39358,145.043318,38.748070,106.295247,15.233526,1
10,485,take,24943,159.405726,36.702828,122.702898,13.813832,1
21,920,give+take,53473,130.916908,37.453318,93.463590,13.427606,2
0,485,give,14415,120.191320,39.133202,81.058118,11.646548,1
11,920,take,53473,130.916908,34.637586,96.279322,10.877704,2
22,401,give+take,3711,122.979578,48.724090,74.255488,10.724268,3
1,401,give,2265,116.905201,44.586908,72.318293,10.412483,2
23,830,give+take,9132,114.171679,44.919886,69.251794,10.020051,4
12,401,take,1446,132.494421,44.415267,88.079154,9.966520,3
2,781,give,21280,105.721673,37.985412,67.736261,9.765499,3


**Outliers: tokens exchanged vs mean intrinsic-dimension gap (K-means partition)**

,cluster,stratum,tokens_exchanged,mean_intrinsic_dim_gap_km,mean_intrinsic_dim_gap_km_expected,mean_intrinsic_dim_gap_km_residual,mean_intrinsic_dim_gap_km_outlier_score,mean_intrinsic_dim_gap_km_rank
10,234,take,134,-534.126866,151.255084,-685.381950,6.855978,1
11,454,take,67384,549.292948,56.117726,493.175222,5.036772,2
20,454,give+take,72235,513.934630,-6.031455,519.966085,4.717399,1
12,62,take,465,-329.982796,132.225578,-462.208374,4.603947,3
21,897,give+take,48428,-483.386037,1.061389,-484.447426,4.572588,2
0,897,give,48428,-483.386037,-48.113825,-435.272212,4.529533,1
1,62,give,9645,-430.729705,-1.975276,-428.754429,4.463089,2
13,84,take,365,569.326027,135.929014,433.397014,4.433554,4
14,491,take,4369,-338.335775,97.961818,-436.297593,4.342483,5
2,485,give,14415,-430.036559,-13.464738,-416.571821,4.338895,3


**Outliers appearing in both K-means plots**

,cluster,stratum,tokens_exchanged,mean_centroid_distance_km,mean_centroid_distance_km_expected,mean_centroid_distance_km_residual,mean_centroid_distance_km_outlier_score,mean_centroid_distance_km_rank,mean_intrinsic_dim_gap_km,mean_intrinsic_dim_gap_km_expected,mean_intrinsic_dim_gap_km_residual,mean_intrinsic_dim_gap_km_outlier_score,mean_intrinsic_dim_gap_km_rank,joint_outlier_score
0,485,take,24943,159.405726,36.702828,122.702898,13.813832,1,-304.694263,71.317720,-376.011983,3.734144,6,17.547977
1,485,give,14415,120.191320,39.133202,81.058118,11.646548,1,-430.036559,-13.464738,-416.571821,4.338895,3,15.985443
2,781,give,21280,105.721673,37.985412,67.736261,9.765499,3,-440.444549,-24.601611,-415.842937,4.331465,4,14.096963
3,802,give,31737,102.703763,36.807513,65.896250,9.505689,4,-433.886347,-36.030624,-397.855723,4.148097,7,13.653785
4,781,give+take,21280,105.721673,41.345926,64.375748,9.333800,5,-440.444549,15.648286,-456.092835,4.310332,4,13.644131
5,802,give+take,31737,102.703763,39.657277,63.046486,9.146720,6,-433.886347,8.557766,-442.444113,4.184092,7,13.330813
6,897,give,48428,87.511634,35.562192,51.949442,7.536395,8,-483.386037,-48.113825,-435.272212,4.529533,1,12.065929
7,897,give+take,48428,87.511634,37.871972,49.639662,7.259851,9,-483.386037,1.061389,-484.447426,4.572588,2,11.832440
8,62,take,465,104.995594,47.487791,57.507804,6.569501,7,-329.982796,132.225578,-462.208374,4.603947,3,11.173448
9,84,take,365,98.444181,48.143558,50.300622,5.768655,8,569.326027,135.929014,433.397014,4.433554,4,10.202209


## 5. High-Churn vs Low-Churn Spectral Change

Use the PCA eigenvalue spectra already saved in the K-means and MFA
`intrinsic_dims.pt` artifacts. Clusters are eligible only when both artifacts
contain a nonempty spectrum estimated from exactly the same number of sampled
activations. No assignments or activation shards are loaded here.

High-churn clusters are the top 10% by
`relative churn = (left + joined) / K-means size`. Controls are selected without
replacement from the bottom half by churn and matched by K-means cluster size.
From each normalized saved spectrum `p`, report:

- intrinsic dimension at the saved variance threshold (normally 90%);
- effective rank, `exp(-sum(p * log(p)))`;
- variance explained by the first 10 PCs;
- spectral isotropy, defined here as normalized participation ratio
  `1 / (len(p) * sum(p**2))` (1 is a flat spectrum);
- normalized area between the K-means and MFA cumulative-variance curves.

Changes are `MFA - K-means`. Thus negative dimension/effective-rank/isotropy
together with positive first-10-PC variance means the trained membership is more
spectrally concentrated; the opposite pattern means it is more diffuse. Confidence
intervals below bootstrap the size-matched cluster pairs, treating each saved PCA
spectrum as one cluster-level estimate.


In [15]:
from scipy.optimize import linear_sum_assignment

HIGH_CHURN_FRACTION = 0.10
LOW_CHURN_POOL_FRACTION = 0.50

SPECTRAL_ID_PATHS = {
    "km": CENTROIDS_DIR / "intrinsic_dims.pt",
    "mfa": MFA_RUN / "intrinsic_dims.pt",
}
id_data = {
    label: torch.load(path, map_location="cpu", weights_only=True)
    for label, path in SPECTRAL_ID_PATHS.items()
}
km_id, mfa_id = id_data["km"], id_data["mfa"]
assert int(km_id["K"]) == int(mfa_id["K"]) == K
assert np.isclose(km_id["variance_threshold"], mfa_id["variance_threshold"])
ID_THRESHOLD = float(km_id["variance_threshold"])

km_sample_sizes = km_id["sample_sizes"].numpy()
mfa_sample_sizes = mfa_id["sample_sizes"].numpy()
valid = np.array([
    km_sample_sizes[k] == mfa_sample_sizes[k]
    and km_sample_sizes[k] >= 2
    and km_id["cluster_variances"][k].numel() > 0
    and km_id["cluster_variances"][k].numel() == mfa_id["cluster_variances"][k].numel()
    for k in range(K)
])

def normalized_spectrum(values: torch.Tensor) -> np.ndarray:
    p = values.double().clamp_min(0).numpy()
    total = p.sum()
    if not np.isfinite(total) or total <= 0:
        raise ValueError("PCA spectrum has non-positive or non-finite total variance")
    return p / total

def spectrum_metrics(p: np.ndarray) -> dict[str, float]:
    cumulative = np.cumsum(p)
    positive = p > 0
    entropy = -(p[positive] * np.log(p[positive])).sum()
    return {
        "intrinsic_dim": float(np.searchsorted(cumulative, ID_THRESHOLD) + 1),
        "effective_rank": float(np.exp(entropy)),
        "var_first10": float(p[:10].sum()),
        "isotropy": float(1.0 / (len(p) * np.square(p).sum())),
    }

rows = []
for cluster in np.flatnonzero(valid):
    p_km = normalized_spectrum(km_id["cluster_variances"][cluster])
    p_mfa = normalized_spectrum(mfa_id["cluster_variances"][cluster])
    km_metrics = spectrum_metrics(p_km)
    mfa_metrics = spectrum_metrics(p_mfa)
    assert int(km_metrics["intrinsic_dim"]) == int(km_id["intrinsic_dims"][cluster])
    assert int(mfa_metrics["intrinsic_dim"]) == int(mfa_id["intrinsic_dims"][cluster])
    curve_gap = np.r_[0.0, np.abs(np.cumsum(p_km) - np.cumsum(p_mfa))]
    row = {
        "cluster": int(cluster),
        "pca_sample_n": int(km_sample_sizes[cluster]),
        "cumulative_variance_area": float(
            np.trapezoid(curve_gap, dx=1.0 / (len(curve_gap) - 1))
        ),
    }
    for metric in km_metrics:
        row[f"km_{metric}"] = km_metrics[metric]
        row[f"mfa_{metric}"] = mfa_metrics[metric]
        row[f"delta_{metric}"] = mfa_metrics[metric] - km_metrics[metric]
    rows.append(row)

spectral_by_cluster = per_cluster.merge(pd.DataFrame(rows), on="cluster", how="inner")
SPECTRAL_CHANGE_METRICS = [
    "delta_intrinsic_dim",
    "delta_effective_rank",
    "delta_var_first10",
    "delta_isotropy",
    "cumulative_variance_area",
]

n_group = max(1, int(np.ceil(HIGH_CHURN_FRACTION * len(spectral_by_cluster))))
ranked = spectral_by_cluster.sort_values(["rel_churn", "cluster"])
high = ranked.tail(n_group).copy()
low_pool = ranked.head(int(np.ceil(LOW_CHURN_POOL_FRACTION * len(ranked)))).copy()
assert set(high["cluster"]).isdisjoint(low_pool["cluster"])

cost = np.abs(
    np.log1p(high["km_size"].to_numpy())[:, None]
    - np.log1p(low_pool["km_size"].to_numpy())[None, :]
)
# Enforce the same saved PCA sample count across every matched high/low pair.
cost += 1e6 * (
    high["pca_sample_n"].to_numpy()[:, None]
    != low_pool["pca_sample_n"].to_numpy()[None, :]
)
high_row, low_row = linear_sum_assignment(cost)
high_selected = high.iloc[high_row].copy().reset_index(drop=True)
low_selected = low_pool.iloc[low_row].copy().reset_index(drop=True)
assert np.array_equal(high_selected["pca_sample_n"], low_selected["pca_sample_n"])

matches = pd.DataFrame({
    "match_id": np.arange(n_group),
    "high_cluster": high_selected["cluster"],
    "low_cluster": low_selected["cluster"],
    "high_rel_churn": high_selected["rel_churn"],
    "low_rel_churn": low_selected["rel_churn"],
    "high_km_size": high_selected["km_size"],
    "low_km_size": low_selected["km_size"],
    "pca_sample_n": high_selected["pca_sample_n"],
})
matches["log_size_gap"] = np.abs(
    np.log1p(matches["high_km_size"]) - np.log1p(matches["low_km_size"])
)
high_selected["match_id"], high_selected["group"] = matches["match_id"], "high"
low_selected["match_id"], low_selected["group"] = matches["match_id"], "low"
matched_spectral = pd.concat([high_selected, low_selected], ignore_index=True)

print(
    f"Equal-sample valid spectra: {len(spectral_by_cluster)}/{K}; "
    f"high/low matched clusters: {n_group} each"
)
display(matches.describe(percentiles=[0.1, 0.5, 0.9]))


Equal-sample valid spectra: 600/1000; high/low matched clusters: 60 each


,match_id,high_cluster,low_cluster,high_rel_churn,low_rel_churn,high_km_size,low_km_size,pca_sample_n,log_size_gap
count,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.0,60.000000
mean,29.500000,484.083333,468.383333,7.165495,0.447006,41927.100000,41963.916667,10000.0,0.015174
std,17.464249,284.710873,298.474431,4.511237,0.295137,30165.080065,30080.607730,0.0,0.027417
min,0.000000,0.000000,30.000000,3.547864,0.015352,10064.000000,10964.000000,10000.0,0.000026
10%,5.900000,78.500000,76.600000,3.687167,0.074266,16460.200000,16686.200000,10000.0,0.000512
50%,29.500000,479.000000,426.500000,5.520887,0.440363,35377.000000,34996.500000,10000.0,0.004632
90%,53.100000,836.300000,872.700000,11.479359,0.870189,67691.300000,67433.200000,10000.0,0.037446
max,59.000000,970.000000,998.000000,27.126710,1.037751,180327.000000,180789.000000,10000.0,0.122595


### 5.1 Are Low Top-q K-Means Spectra Also High-Churn?

Reproduce the left-tail selection from `kmeans_mfa_comparison.ipynb`: the
bottom 10% of K-means clusters by variance fraction captured in the first
`q=Q` PCs. Compare those cluster IDs with the high-churn group already defined
above (top 10% by relative churn). The overlap is reported against its random
expectation, with a one-sided Fisher exact test for enrichment.


In [16]:
from scipy.stats import fisher_exact

KMEANS_LEFT_TAIL_FRACTION = 0.10
comparison_valid = (km_id["intrinsic_dims"] > 0) & (mfa_id["intrinsic_dims"] > 0)
kmeans_topq_rows = []
for cluster in torch.nonzero(comparison_valid, as_tuple=True)[0].tolist():
    p_km = normalized_spectrum(km_id["cluster_variances"][cluster])
    kmeans_topq_rows.append(
        {"cluster": cluster, "km_var_first10": float(p_km[:Q].sum())}
    )
kmeans_topq_by_cluster = pd.DataFrame(kmeans_topq_rows)
kmeans_left_tail_threshold = float(
    kmeans_topq_by_cluster["km_var_first10"].quantile(KMEANS_LEFT_TAIL_FRACTION)
)
kmeans_left_tail = kmeans_topq_by_cluster.loc[
    kmeans_topq_by_cluster["km_var_first10"] <= kmeans_left_tail_threshold
].merge(per_cluster[["cluster", "rel_churn"]], on="cluster", how="left")
kmeans_left_tail = kmeans_left_tail.sort_values(["km_var_first10", "cluster"])
kmeans_left_tail_clusters = kmeans_left_tail["cluster"].astype(int).tolist()
high_churn_clusters = sorted(high["cluster"].astype(int).tolist())
left_tail_high_churn_clusters = sorted(
    set(kmeans_left_tail_clusters).intersection(high_churn_clusters)
)

eligible_n = len(kmeans_topq_by_cluster)
assert set(high_churn_clusters).issubset(kmeans_topq_by_cluster["cluster"])
overlap_n = len(left_tail_high_churn_clusters)
left_only_n = len(kmeans_left_tail_clusters) - overlap_n
high_only_n = len(high_churn_clusters) - overlap_n
neither_n = eligible_n - overlap_n - left_only_n - high_only_n
tail_churn_contingency = pd.DataFrame(
    [[overlap_n, left_only_n], [high_only_n, neither_n]],
    index=["left spectral tail", "not left spectral tail"],
    columns=["high churn", "not high churn"],
)
odds_ratio, fisher_p = fisher_exact(
    tail_churn_contingency.to_numpy(), alternative="greater"
)
expected_overlap = (
    len(kmeans_left_tail_clusters) * len(high_churn_clusters) / eligible_n
)
enrichment = overlap_n / expected_overlap if expected_overlap > 0 else np.nan

tail_churn_summary = pd.DataFrame(
    [
        {
            "eligible_clusters": eligible_n,
            "left_tail_clusters": len(kmeans_left_tail_clusters),
            "high_churn_clusters": len(high_churn_clusters),
            "overlap": overlap_n,
            "expected_overlap": expected_overlap,
            "enrichment_vs_random": enrichment,
            "left_tail_high_churn_rate": overlap_n / len(kmeans_left_tail_clusters),
            "non_tail_high_churn_rate": high_only_n / (eligible_n - len(kmeans_left_tail_clusters)),
            "fisher_odds_ratio": odds_ratio,
            "fisher_p_greater": fisher_p,
        }
    ]
)
kmeans_left_tail["high_churn"] = kmeans_left_tail["cluster"].isin(
    left_tail_high_churn_clusters
)

print("K-means left-tail clusters that are also high-churn:")
print(left_tail_high_churn_clusters)
display(tail_churn_summary)
display(tail_churn_contingency)
display(
    kmeans_left_tail[
        ["cluster", "km_var_first10", "rel_churn", "high_churn"]
    ].reset_index(drop=True)
)


K-means left-tail clusters that are also high-churn:
[60, 192, 296, 456, 492]


,eligible_clusters,left_tail_clusters,high_churn_clusters,overlap,expected_overlap,enrichment_vs_random,left_tail_high_churn_rate,non_tail_high_churn_rate,fisher_odds_ratio,fisher_p_greater
0,606,61,60,5,6.039604,0.827869,0.081967,0.100917,0.795455,0.74829


,high churn,not high churn
left spectral tail,5,56
not left spectral tail,55,490


,cluster,km_var_first10,rel_churn,high_churn
0,192,0.789532,3.735571,True
1,485,0.802952,2.454200,False
2,456,0.818308,3.550551,True
3,923,0.834208,1.103421,False
4,421,0.847914,3.126755,False
...,...,...,...,...
56,380,0.885572,1.130180,False
57,626,0.885805,0.665068,False
58,30,0.886254,0.138288,False
59,238,0.886797,2.605570,False


In [17]:
BOOTSTRAP_REPS = 10_000
SPECTRAL_SEED = 20260715
rng = np.random.default_rng(SPECTRAL_SEED)

high_values = high_selected[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
low_values = low_selected[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
n_pairs = len(matches)
bootstrap_indices = rng.integers(0, n_pairs, size=(BOOTSTRAP_REPS, n_pairs))

group_comparison_rows = []
for metric_i, metric in enumerate(SPECTRAL_CHANGE_METRICS):
    high_boot = high_values[bootstrap_indices, metric_i].mean(axis=1)
    low_boot = low_values[bootstrap_indices, metric_i].mean(axis=1)
    diff_boot = high_boot - low_boot
    p_two_sided = min(
        1.0, 2 * min(np.mean(diff_boot <= 0), np.mean(diff_boot >= 0))
    )
    group_comparison_rows.append({
        "metric": metric,
        "high_mean": high_values[:, metric_i].mean(),
        "low_mean": low_values[:, metric_i].mean(),
        "high_minus_low": high_values[:, metric_i].mean() - low_values[:, metric_i].mean(),
        "ci_2.5%": np.quantile(diff_boot, 0.025),
        "ci_97.5%": np.quantile(diff_boot, 0.975),
        "bootstrap_p_two_sided": p_two_sided,
    })

group_spectral_comparison = pd.DataFrame(group_comparison_rows)
print(f"Bootstrapped {n_pairs} matched cluster pairs for {BOOTSTRAP_REPS:,} replicates")
display(group_spectral_comparison)


Bootstrapped 60 matched cluster pairs for 10,000 replicates


,metric,high_mean,low_mean,high_minus_low,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,44.416667,-17.683333,62.100000,21.182083,101.001250,0.0028
1,delta_effective_rank,71.123245,-12.385295,83.508540,47.329249,117.517319,0.0000
2,delta_var_first10,-0.127310,0.033140,-0.160450,-0.203822,-0.116603,0.0000
3,delta_isotropy,0.011273,-0.001829,0.013102,0.006576,0.019144,0.0002
4,cumulative_variance_area,0.022057,0.005334,0.016722,0.012470,0.021278,0.0000


In [18]:
REPORT_COLUMNS = [
    "match_id", "group", "cluster", "rel_churn", "km_size",
    "pca_sample_n",
    "km_intrinsic_dim", "mfa_intrinsic_dim", "delta_intrinsic_dim",
    "km_effective_rank", "mfa_effective_rank", "delta_effective_rank",
    "km_var_first10", "mfa_var_first10", "delta_var_first10",
    "km_isotropy", "mfa_isotropy", "delta_isotropy",
    "cumulative_variance_area",
]
display(Markdown("**Matched clusters and their saved-spectrum changes:**"))
display(matched_spectral[REPORT_COLUMNS].sort_values(["match_id", "group"]))
display(Markdown("**High churn minus size-matched low churn:**"))
display(group_spectral_comparison)

if px is not None:
    plot_data = matched_spectral.melt(
        id_vars=["match_id", "group", "cluster"],
        value_vars=SPECTRAL_CHANGE_METRICS,
        var_name="metric", value_name="MFA - K-means",
    )
    fig = px.box(
        plot_data, x="group", y="MFA - K-means", color="group",
        facet_col="metric", facet_col_wrap=2, points="all",
        color_discrete_map={"high": "#d95f02", "low": "#7570b3"},
        category_orders={"group": ["low", "high"]},
        title="Saved-spectrum change: high churn vs size-matched low churn",
        template=PLOT_TEMPLATE,
    )
    fig.update_yaxes(matches=None, showticklabels=True, zeroline=True, zerolinecolor="#0b0b0b")
    fig.update_layout(height=950, width=1100, showlegend=False)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    save_fig(fig, "high_vs_low_churn_spectral_change")
    fig.show()


**Matched clusters and their saved-spectrum changes:**

,match_id,group,cluster,rel_churn,km_size,pca_sample_n,km_intrinsic_dim,mfa_intrinsic_dim,delta_intrinsic_dim,km_effective_rank,mfa_effective_rank,delta_effective_rank,km_var_first10,mfa_var_first10,delta_var_first10,km_isotropy,mfa_isotropy,delta_isotropy,cumulative_variance_area
0,0,high,729,3.547864,32718,10000,234.0,335.0,101.0,97.355572,158.700319,61.344747,0.492824,0.407734,-0.085089,0.010855,0.022257,0.011403,0.013398
60,0,low,166,0.801079,32621,10000,389.0,292.0,-97.0,165.205721,95.404455,-69.801266,0.404409,0.526095,0.121686,0.019594,0.009048,-0.010546,0.013279
1,1,high,456,3.550551,14965,10000,633.0,219.0,-414.0,417.901601,71.425484,-346.476117,0.216189,0.579676,0.363487,0.034431,0.008387,-0.026044,0.069823
61,1,low,697,0.044044,16302,10000,16.0,19.0,3.0,11.471294,12.580938,1.109645,0.854862,0.837683,-0.017178,0.002239,0.002338,0.000099,0.000499
2,2,high,374,3.568021,28506,10000,232.0,320.0,88.0,115.241399,197.998279,82.756881,0.441634,0.333369,-0.108265,0.016045,0.026852,0.010807,0.014105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,57,low,974,0.158593,18475,10000,160.0,177.0,17.0,62.227736,76.915300,14.687564,0.581573,0.543585,-0.037988,0.006918,0.010077,0.003160,0.002736
58,58,high,779,18.946964,17045,10000,402.0,354.0,-48.0,235.965983,256.558306,20.592323,0.305588,0.269775,-0.035812,0.031533,0.027097,-0.004436,0.010400
118,58,low,37,0.265722,17078,10000,71.0,16.0,-55.0,27.340179,10.046834,-17.293345,0.729190,0.868734,0.139544,0.003771,0.002038,-0.001733,0.008968
59,59,high,847,27.126710,15571,10000,250.0,409.0,159.0,98.928376,403.794104,304.865727,0.482935,0.165165,-0.317770,0.009619,0.078143,0.068524,0.041943


**High churn minus size-matched low churn:**

,metric,high_mean,low_mean,high_minus_low,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,44.416667,-17.683333,62.100000,21.182083,101.001250,0.0028
1,delta_effective_rank,71.123245,-12.385295,83.508540,47.329249,117.517319,0.0000
2,delta_var_first10,-0.127310,0.033140,-0.160450,-0.203822,-0.116603,0.0000
3,delta_isotropy,0.011273,-0.001829,0.013102,0.006576,0.019144,0.0002
4,cumulative_variance_area,0.022057,0.005334,0.016722,0.012470,0.021278,0.0000


In [19]:
display(Markdown("**Size-matching diagnostics:**"))
display(matches)
print(
    f"median matched size ratio: {np.exp(matches['log_size_gap'].median()):.3f}; "
    f"90th percentile: {np.exp(matches['log_size_gap'].quantile(0.9)):.3f}"
)


**Size-matching diagnostics:**

,match_id,high_cluster,low_cluster,high_rel_churn,low_rel_churn,high_km_size,low_km_size,pca_sample_n,log_size_gap
0,0,729,166,3.547864,0.801079,32718,32621,10000,0.002969
1,1,456,697,3.550551,0.044044,14965,16302,10000,0.085568
2,2,374,979,3.568021,0.535516,28506,28776,10000,0.009427
3,3,934,822,3.583680,0.979411,35612,35358,10000,0.007158
4,4,509,807,3.629024,0.176969,24511,25010,10000,0.020153
5,5,346,651,3.634052,0.521107,52062,52565,10000,0.009615
6,6,583,353,3.693069,0.424034,29707,29724,10000,0.000572
7,7,60,787,3.714410,0.274102,87468,86840,10000,0.007206
8,8,192,288,3.735571,0.618881,61510,61574,10000,0.001040
9,9,481,205,3.745748,0.096638,16582,16598,10000,0.000964


median matched size ratio: 1.005; 90th percentile: 1.038


### 5.2 Spectral Change Across All Clusters

Section 5 compared only the top-10% churn clusters against size-matched
low-churn controls. Here the same saved-spectrum deltas (`MFA - K-means`) are
reported for **every** equal-sample cluster in `spectral_by_cluster`, to answer
whether training raises or lowers intrinsic dimension overall — not just for
the extreme movers.

> **Why this disagrees with `kmeans_mfa_comparison.ipynb` section 4.1.** That
> notebook loads its K-means IDs from
> `dalg-cache/output/experiments/centroids_1000_05/intrinsic_dims.pt`, which was
> computed with `max_samples=2000`, while its MFA IDs use `max_samples=10000`.
> The ID at a fixed variance threshold grows strongly with PCA sample count
> (this same K-means partition scores mean ID ≈ 247 at 2k samples vs ≈ 308 at
> 10k), so the apparent ΔID ≈ +8 there is a sample-size artifact, not a
> geometric effect. With matched 10k-sample artifacts (used here), ΔID is
> clearly negative.

In [20]:
SPECTRAL_SEED_ALL = 20260716
rng_all = np.random.default_rng(SPECTRAL_SEED_ALL)
boot_idx_all = rng_all.integers(
    0, len(spectral_by_cluster), size=(BOOTSTRAP_REPS, len(spectral_by_cluster))
)

all_cluster_rows = []
for metric in SPECTRAL_CHANGE_METRICS:
    values = spectral_by_cluster[metric].to_numpy(dtype=np.float64)
    boot_means = values[boot_idx_all].mean(axis=1)
    all_cluster_rows.append({
        "metric": metric,
        "mean": values.mean(),
        "median": np.median(values),
        "frac_negative": float((values < 0).mean()),
        "frac_positive": float((values > 0).mean()),
        "ci_2.5%": np.quantile(boot_means, 0.025),
        "ci_97.5%": np.quantile(boot_means, 0.975),
    })
all_cluster_spectral = pd.DataFrame(all_cluster_rows)
print(f"All equal-sample clusters: {len(spectral_by_cluster)}/{K}")
display(all_cluster_spectral)

spectral_by_cluster["churn_quintile"] = pd.qcut(
    spectral_by_cluster["rel_churn"], 5, labels=[f"Q{i}" for i in range(1, 6)]
)
quintile_table = spectral_by_cluster.groupby("churn_quintile", observed=True).agg(
    n=("cluster", "size"),
    rel_churn_median=("rel_churn", "median"),
    delta_id_mean=("delta_intrinsic_dim", "mean"),
    delta_id_median=("delta_intrinsic_dim", "median"),
    frac_id_negative=("delta_intrinsic_dim", lambda x: (x < 0).mean()),
    delta_effective_rank_mean=("delta_effective_rank", "mean"),
)
display(Markdown("**ΔID by relative-churn quintile (Q1 = lowest churn):**"))
display(quintile_table)

from scipy.stats import spearmanr

rho_all, p_all = spearmanr(
    spectral_by_cluster["rel_churn"], spectral_by_cluster["delta_intrinsic_dim"]
)
print(f"Spearman(rel_churn, delta_intrinsic_dim): rho={rho_all:.3f}, p={p_all:.2e}")

if px is not None:
    fig = px.histogram(
        spectral_by_cluster,
        x="delta_intrinsic_dim",
        nbins=60,
        title="ΔID = ID(mfa) − ID(kmeans), all equal-sample clusters (matched 10k-sample artifacts)",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
    )
    fig.add_vline(x=0, line_dash="dash", line_color="#0b0b0b")
    fig.update_layout(xaxis_title="ΔID (trained − init)", yaxis_title="clusters", bargap=0.05)
    save_fig(fig, "delta_id_hist_all_clusters")
    fig.show()

    fig = px.scatter(
        spectral_by_cluster,
        x="rel_churn",
        y="delta_intrinsic_dim",
        hover_data=["cluster", "km_size", "km_intrinsic_dim", "mfa_intrinsic_dim"],
        title="ΔID vs relative churn, all equal-sample clusters",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
        opacity=0.55,
    )
    fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
    fig.update_layout(xaxis_title="relative churn (left + joined) / km size", yaxis_title="ΔID")
    save_fig(fig, "delta_id_vs_rel_churn_all_clusters")
    fig.show()

All equal-sample clusters: 600/1000


,metric,mean,median,frac_negative,frac_positive,ci_2.5%,ci_97.5%
0,delta_intrinsic_dim,-24.596667,-20.000000,0.636667,0.360000,-32.276875,-17.158333
1,delta_effective_rank,-10.985309,-13.546666,0.671667,0.328333,-16.690567,-5.233307
2,delta_var_first10,0.016322,0.029404,0.326667,0.673333,0.007378,0.025134
3,delta_isotropy,-0.001440,-0.001529,0.648333,0.351667,-0.002422,-0.000478
4,cumulative_variance_area,0.009881,0.006524,0.000000,1.000000,0.009051,0.010761


**ΔID by relative-churn quintile (Q1 = lowest churn):**

,n,rel_churn_median,delta_id_mean,delta_id_median,frac_id_negative,delta_effective_rank_mean
churn_quintile,,,,,,
Q1,120,0.170602,-18.825000,-15.0,0.750000,-11.871266
Q2,120,0.616811,-44.258333,-37.0,0.791667,-27.310739
Q3,120,1.039031,-58.183333,-52.0,0.750000,-38.380513
Q4,120,1.685612,-28.200000,-12.5,0.583333,-18.695524
Q5,120,3.536543,26.483333,26.0,0.308333,41.331496


Spearman(rel_churn, delta_intrinsic_dim): rho=0.220, p=5.04e-08


## 6. Section 4.6 Outliers vs Size-Matched Controls

Repeat the saved-spectrum analysis using the outliers identified in Section 4.6
instead of selecting clusters by relative churn. An **outlier cluster** is any
unique cluster appearing in either `distance_outliers` or `id_gap_outliers`, in
any of the `give`, `take`, or `give+take` strata. The stricter
`overlap_outliers` table contains only one unique cluster (two cluster-stratum
rows), which is too small for a group comparison.

Controls exclude every Section 4.6 outlier and are matched without replacement by
K-means cluster size and saved PCA sample count. The spectral changes and
matched-pair bootstrap are otherwise identical to Section 5.


In [21]:
outlier_events = pd.concat(
    [
        distance_outliers[["cluster", "stratum"]].assign(
            outlier_source="centroid_distance"
        ),
        id_gap_outliers[["cluster", "stratum"]].assign(
            outlier_source="intrinsic_dim_gap"
        ),
    ],
    ignore_index=True,
).drop_duplicates()
outlier_catalog = (
    outlier_events.groupby("cluster")
    .agg(
        outlier_sources=("outlier_source", lambda x: ", ".join(sorted(set(x)))),
        outlier_strata=("stratum", lambda x: ", ".join(sorted(set(x)))),
    )
    .reset_index()
)
all_outlier_ids = set(outlier_catalog["cluster"].astype(int))
outlier_selected = spectral_by_cluster.merge(outlier_catalog, on="cluster", how="inner")
control_pool = spectral_by_cluster[
    ~spectral_by_cluster["cluster"].isin(all_outlier_ids)
].copy()
assert len(outlier_selected) > 1
assert len(control_pool) >= len(outlier_selected)

outlier_cost = np.abs(
    np.log1p(outlier_selected["km_size"].to_numpy())[:, None]
    - np.log1p(control_pool["km_size"].to_numpy())[None, :]
)
outlier_cost += 1e6 * (
    outlier_selected["pca_sample_n"].to_numpy()[:, None]
    != control_pool["pca_sample_n"].to_numpy()[None, :]
)
outlier_row, control_row = linear_sum_assignment(outlier_cost)
outlier_selected = outlier_selected.iloc[outlier_row].copy().reset_index(drop=True)
outlier_controls = control_pool.iloc[control_row].copy().reset_index(drop=True)
assert np.array_equal(outlier_selected["pca_sample_n"], outlier_controls["pca_sample_n"])

n_outlier_pairs = len(outlier_selected)
outlier_matches = pd.DataFrame({
    "match_id": np.arange(n_outlier_pairs),
    "outlier_cluster": outlier_selected["cluster"],
    "control_cluster": outlier_controls["cluster"],
    "outlier_sources": outlier_selected["outlier_sources"],
    "outlier_strata": outlier_selected["outlier_strata"],
    "outlier_rel_churn": outlier_selected["rel_churn"],
    "control_rel_churn": outlier_controls["rel_churn"],
    "outlier_km_size": outlier_selected["km_size"],
    "control_km_size": outlier_controls["km_size"],
    "pca_sample_n": outlier_selected["pca_sample_n"],
})
outlier_matches["log_size_gap"] = np.abs(
    np.log1p(outlier_matches["outlier_km_size"])
    - np.log1p(outlier_matches["control_km_size"])
)
outlier_selected["match_id"], outlier_selected["group"] = (
    outlier_matches["match_id"], "outlier"
)
outlier_controls["match_id"], outlier_controls["group"] = (
    outlier_matches["match_id"], "control"
)
outlier_controls["outlier_sources"] = "not an outlier"
outlier_controls["outlier_strata"] = ""
matched_outlier_spectral = pd.concat(
    [outlier_selected, outlier_controls], ignore_index=True
)

print(
    f"Section 4.6 unique outlier clusters: {len(all_outlier_ids)}; "
    f"with equal-sample spectra: {n_outlier_pairs}; excluded: "
    f"{len(all_outlier_ids) - n_outlier_pairs}"
)
display(outlier_matches.describe(percentiles=[0.1, 0.5, 0.9]))


Section 4.6 unique outlier clusters: 27; with equal-sample spectra: 7; excluded: 20


,match_id,outlier_cluster,control_cluster,outlier_rel_churn,control_rel_churn,outlier_km_size,control_km_size,pca_sample_n,log_size_gap
count,7.000000,7.000000,7.00000,7.000000,7.000000,7.000000,7.000000,7.0,7.000000
mean,3.000000,361.857143,681.00000,4.099962,6.511211,40259.000000,40132.000000,10000.0,0.006062
std,2.160247,235.373766,320.77926,6.638883,9.318368,46100.245148,46049.797321,0.0,0.010477
min,0.000000,7.000000,63.00000,0.044044,0.044354,14965.000000,14893.000000,10000.0,0.000000
10%,0.600000,118.000000,313.80000,0.075600,1.376520,15608.200000,15299.800000,10000.0,0.000507
50%,3.000000,456.000000,790.00000,2.454200,2.852766,16598.000000,16582.000000,10000.0,0.002262
90%,5.400000,573.400000,958.40000,9.719373,14.990897,91952.400000,91859.200000,10000.0,0.014688
max,6.000000,697.000000,968.00000,18.695076,27.126710,137616.000000,137305.000000,10000.0,0.029486


In [22]:
OUTLIER_BOOTSTRAP_REPS = 10_000
OUTLIER_SPECTRAL_SEED = 20260716
outlier_rng = np.random.default_rng(OUTLIER_SPECTRAL_SEED)

outlier_values = outlier_selected[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
control_values = outlier_controls[SPECTRAL_CHANGE_METRICS].to_numpy(dtype=np.float64)
outlier_bootstrap_indices = outlier_rng.integers(
    0, n_outlier_pairs, size=(OUTLIER_BOOTSTRAP_REPS, n_outlier_pairs)
)

outlier_comparison_rows = []
for metric_i, metric in enumerate(SPECTRAL_CHANGE_METRICS):
    outlier_boot = outlier_values[outlier_bootstrap_indices, metric_i].mean(axis=1)
    control_boot = control_values[outlier_bootstrap_indices, metric_i].mean(axis=1)
    diff_boot = outlier_boot - control_boot
    p_two_sided = min(
        1.0, 2 * min(np.mean(diff_boot <= 0), np.mean(diff_boot >= 0))
    )
    outlier_comparison_rows.append({
        "metric": metric,
        "outlier_mean": outlier_values[:, metric_i].mean(),
        "control_mean": control_values[:, metric_i].mean(),
        "outlier_minus_control": (
            outlier_values[:, metric_i].mean() - control_values[:, metric_i].mean()
        ),
        "ci_2.5%": np.quantile(diff_boot, 0.025),
        "ci_97.5%": np.quantile(diff_boot, 0.975),
        "bootstrap_p_two_sided": p_two_sided,
    })

outlier_spectral_comparison = pd.DataFrame(outlier_comparison_rows)
print(
    f"Bootstrapped {n_outlier_pairs} outlier/control pairs for "
    f"{OUTLIER_BOOTSTRAP_REPS:,} replicates"
)
display(outlier_spectral_comparison)


Bootstrapped 7 outlier/control pairs for 10,000 replicates


,metric,outlier_mean,control_mean,outlier_minus_control,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,-153.000000,53.142857,-206.142857,-437.857143,4.050000,0.0518
1,delta_effective_rank,-125.930078,59.123720,-185.053798,-434.398630,47.070415,0.1266
2,delta_var_first10,0.075533,-0.102836,0.178368,-0.030060,0.398766,0.1004
3,delta_isotropy,-0.022813,0.012242,-0.035056,-0.082694,0.007675,0.1344
4,cumulative_variance_area,0.043320,0.012373,0.030946,0.007829,0.052684,0.0100


In [23]:
OUTLIER_REPORT_COLUMNS = [
    "match_id", "group", "cluster", "outlier_sources", "outlier_strata",
    "rel_churn", "km_size", "pca_sample_n",
    "km_intrinsic_dim", "mfa_intrinsic_dim", "delta_intrinsic_dim",
    "km_effective_rank", "mfa_effective_rank", "delta_effective_rank",
    "km_var_first10", "mfa_var_first10", "delta_var_first10",
    "km_isotropy", "mfa_isotropy", "delta_isotropy",
    "cumulative_variance_area",
]
display(Markdown("**Section 4.6 outliers and their size-matched controls:**"))
display(
    matched_outlier_spectral[OUTLIER_REPORT_COLUMNS]
    .sort_values(["match_id", "group"])
)
display(Markdown("**Outliers minus size-matched controls:**"))
display(outlier_spectral_comparison)
print(
    f"median matched size ratio: "
    f"{np.exp(outlier_matches['log_size_gap'].median()):.3f}; "
    f"90th percentile: {np.exp(outlier_matches['log_size_gap'].quantile(0.9)):.3f}"
)

if px is not None:
    outlier_plot_data = matched_outlier_spectral.melt(
        id_vars=["match_id", "group", "cluster"],
        value_vars=SPECTRAL_CHANGE_METRICS,
        var_name="metric", value_name="MFA - K-means",
    )
    fig = px.box(
        outlier_plot_data, x="group", y="MFA - K-means", color="group",
        facet_col="metric", facet_col_wrap=2, points="all",
        color_discrete_map={"outlier": "#d95f02", "control": "#7570b3"},
        category_orders={"group": ["control", "outlier"]},
        title="Saved-spectrum change: Section 4.6 outliers vs size-matched controls",
        template=PLOT_TEMPLATE,
    )
    fig.update_yaxes(
        matches=None, showticklabels=True, zeroline=True, zerolinecolor="#0b0b0b"
    )
    fig.update_layout(height=950, width=1100, showlegend=False)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    save_fig(fig, "section46_outliers_spectral_change")
    fig.show()


**Section 4.6 outliers and their size-matched controls:**

,match_id,group,cluster,outlier_sources,outlier_strata,rel_churn,km_size,pca_sample_n,km_intrinsic_dim,mfa_intrinsic_dim,...,km_effective_rank,mfa_effective_rank,delta_effective_rank,km_var_first10,mfa_var_first10,delta_var_first10,km_isotropy,mfa_isotropy,delta_isotropy,cumulative_variance_area
7,0,control,952,not an outlier,,2.643915,18709,10000,81.0,177.0,...,44.620347,89.626956,45.006609,0.639490,0.516322,-0.123167,0.006590,0.013116,0.006526,0.012815
0,0,outlier,7,"centroid_distance, intrinsic_dim_gap",take,18.695076,18785,10000,176.0,504.0,...,62.861034,420.100479,357.239444,0.593096,0.182485,-0.410611,0.007091,0.059233,0.052141,0.062389
8,1,control,666,not an outlier,,2.852766,61562,10000,215.0,219.0,...,77.383112,121.298539,43.915427,0.563654,0.424215,-0.139440,0.009093,0.017726,0.008634,0.005347
1,1,outlier,192,intrinsic_dim_gap,"give, take",3.735571,61510,10000,700.0,313.0,...,626.982085,212.062032,-414.920053,0.123581,0.292300,0.168719,0.139394,0.026141,-0.113254,0.069753
9,2,control,481,not an outlier,,3.745748,16582,10000,250.0,290.0,...,101.087145,104.905676,3.818532,0.497811,0.505420,0.007609,0.011728,0.011902,0.000174,0.004604
2,2,outlier,205,intrinsic_dim_gap,give,0.096638,16598,10000,149.0,163.0,...,56.339742,57.717644,1.377902,0.618059,0.610848,-0.007210,0.007128,0.006682,-0.000446,0.001559
10,3,control,790,not an outlier,,6.900356,14893,10000,186.0,197.0,...,78.086635,75.748744,-2.337891,0.554554,0.565724,0.011170,0.009923,0.010219,0.000296,0.005361
3,3,outlier,456,intrinsic_dim_gap,take,3.550551,14965,10000,633.0,219.0,...,417.901601,71.425484,-346.476117,0.216189,0.579676,0.363487,0.034431,0.008387,-0.026044,0.069823
11,4,control,847,not an outlier,,27.126710,15571,10000,250.0,409.0,...,98.928376,403.794104,304.865727,0.482935,0.165165,-0.317770,0.009619,0.078143,0.068524,0.041943
4,4,outlier,485,"centroid_distance, intrinsic_dim_gap","give, give+take, take",2.454200,16037,10000,658.0,104.0,...,475.522314,43.444681,-432.077633,0.201460,0.580056,0.378596,0.069549,0.003900,-0.065649,0.090246


**Outliers minus size-matched controls:**

,metric,outlier_mean,control_mean,outlier_minus_control,ci_2.5%,ci_97.5%,bootstrap_p_two_sided
0,delta_intrinsic_dim,-153.000000,53.142857,-206.142857,-437.857143,4.050000,0.0518
1,delta_effective_rank,-125.930078,59.123720,-185.053798,-434.398630,47.070415,0.1266
2,delta_var_first10,0.075533,-0.102836,0.178368,-0.030060,0.398766,0.1004
3,delta_isotropy,-0.022813,0.012242,-0.035056,-0.082694,0.007675,0.1344
4,cumulative_variance_area,0.043320,0.012373,0.030946,0.007829,0.052684,0.0100


median matched size ratio: 1.002; 90th percentile: 1.015


## 7. Summary

Compact table of the headline numbers.


In [24]:
summary = pd.DataFrame(
    [
        {"metric": "same-id agreement", "value": same_id_agreement},
        {"metric": "tokens changing cluster", "value": int(N - shared.sum())},
        {"metric": "median per-cluster churn", "value": float(per_cluster["churn"].median())},
        {"metric": "median relative churn", "value": float(per_cluster["rel_churn"].median())},
        {"metric": "clusters with rel_churn > 1", "value": int((per_cluster["rel_churn"] > 1).sum())},
        {"metric": "clusters that grew (net_delta > 0)", "value": int((per_cluster["net_delta"] > 0).sum())},
        {"metric": "clusters that shrank (net_delta < 0)", "value": int((per_cluster["net_delta"] < 0).sum())},
        {"metric": "max churn cluster", "value": int(per_cluster.loc[per_cluster["churn"].idxmax(), "cluster"])},
    ]
)
display(summary)


,metric,value
0,same-id agreement,4.070964e-01
1,tokens changing cluster,4.368984e+07
2,median per-cluster churn,6.052700e+04
3,median relative churn,1.000000e+00
4,clusters with rel_churn > 1,3.150000e+02
5,clusters that grew (net_delta > 0),4.570000e+02
6,clusters that shrank (net_delta < 0),5.430000e+02
7,max churn cluster,8.330000e+02
